In [1]:
import os
import glob
import requests
import chromadb
from chromadb.utils import embedding_functions

In [2]:
CLASS_FILES_DIR = "class_files"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "gemma3:12b"
OLLAMA_BASE_URL = "http://localhost:11434"  # Default Ollama URL
EMBEDDING_DIMENSION = 768  # Dimension of nomic-embed-text embeddings (adjust if needed)
CHROMA_PERSIST_DIR = "chroma_db"
COLLECTION_NAME = "class_files_collection"
N_RESULTS = 10

In [3]:
def read_rst_file(filepath):
    """Reads an RST file and returns its content."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return None

def get_embedding(text):
    """Gets the embedding for the given text using Ollama."""
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    data = {
        "model": EMBEDDING_MODEL,
        "prompt": text
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        result = response.json()
        return result['embedding']
    except requests.exceptions.RequestException as e:
        print(f"Error getting embedding from Ollama: {e}")
        return None

def query_llm(prompt):
    """Queries the LLM using Ollama."""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    data = {
        "model": LLM_MODEL,
        "prompt": prompt,
        "stream": False  # Set to True for streaming output
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        result = response.json()
        return result['response']
    except requests.exceptions.RequestException as e:
        print(f"Error querying LLM from Ollama: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=100):
    """Simple text chunking function."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks

def create_chroma_client(persist_directory=CHROMA_PERSIST_DIR):
    """Creates and returns a ChromaDB client."""
    return chromadb.PersistentClient(path=persist_directory)

def get_chroma_collection(client, collection_name=COLLECTION_NAME):
    """Gets or creates a ChromaDB collection."""
    return client.get_or_create_collection(name=collection_name)

In [4]:
# Create ChromaDB client and collection
client = create_chroma_client()
collection = get_chroma_collection(client)

# Index the RST files
if not os.path.exists(CLASS_FILES_DIR):
    print(f"Error: Directory '{CLASS_FILES_DIR}' not found. Please create it and add your RST files.")
else:
    rst_files = glob.glob(os.path.join(CLASS_FILES_DIR, "*.rst"))
    if not rst_files:
        print(f"No RST files found in '{CLASS_FILES_DIR}'.")
    else:
        print("Indexing RST files...")
        for filepath in rst_files:
            filename = os.path.basename(filepath)
            content = read_rst_file(filepath)
            if content:
                chunks = chunk_text(content)
                for i, chunk in enumerate(chunks):
                    embedding = get_embedding(chunk)
                    if embedding:
                        doc_id = f"{filename}_chunk_{i}"
                        collection.add(
                            ids=[doc_id],
                            embeddings=[embedding],
                            documents=[chunk],
                            metadatas={"source": filename, "chunk": i}
                        )
        print("Indexing complete.")

Indexing RST files...
Indexing complete.


In [5]:
query = input("Ask a question about the class files: ")

# Get embedding for the query
query_embedding = get_embedding(query)
retrieved_chunks = None

if query_embedding:
    # Search ChromaDB for relevant documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=N_RESULTS
    )

    if results and results['documents'] and results['documents'][0]:
        retrieved_chunks = results['documents'][0]
        print("\nRetrieved Chunks:")
        for i, chunk in enumerate(retrieved_chunks):
            print(f"--- Chunk {i+1} ---")
            print(chunk)
        context = "\n\n".join(retrieved_chunks)
    else:
        print("No relevant documents found for your query.")
else:
    print("Could not generate embedding for your query.")


Retrieved Chunks:
--- Chunk 1 ---
Following the Line: Proportional Control

**Daily Goals**

* Understand the concept of proportional control in robotics and how it differs from on-off control.
* Learn how to implement proportional control for line following by adjusting motor speeds based on the error between sensor readings.
* Understand the role of the proportional gain (KP) and how it affects the robot's line following behavior.
* Learn how to tune the KP value through experimentation to achieve smooth and stable line following.
* Implement a "racing around a circle" activity using proportional control and the line sensor to detect an intersection and turn around.

Introduction to Proportional Control
------------------------------------

In the previous lesson, we designed an *on-off* controller which had discrete actions depending on the error between the two sensors. However, this controller is not very smooth and can lead to oscillations. In this lesson, we will introduc
--- C

In [6]:
if retrieved_chunks:
    # Formulate the prompt for the LLM
    prompt = f"""You are a helpful assistant. Use the following context from class files to answer the user's question. If you don't know the answer, just say you don't know.

    Context:
    {context}

    Question: {query}"""

    # Query the LLM
    print("\nGenerating answer...")
    answer = query_llm(prompt)
    if answer:
        print("\nAnswer:")
        print(answer)
    else:
        print("Could not get an answer from the LLM.")


Generating answer...

Answer:
```python
class LineSensor:  # Mock LineSensor class for demonstration
    def get_error(self):
        """Returns a mock error value.  Replace with actual sensor reading logic."""
        # This is a placeholder, in a real implementation, this would read
        # from the reflectance sensors.  For testing, it just returns a random
        # value.
        import random
        return random.uniform(-1, 1) # Simulate a small error

class Drivetrain: # Mock Drivetrain class for demonstration
    def set_speed(self, left_speed, right_speed):
        """Sets the left and right motor speeds.  Replace with actual motor control logic."""
        print(f"Setting left speed to {left_speed}, right speed to {right_speed}")


class LineTracker:
    def __init__(self, drivetrain):
        self.sensor = LineSensor()
        self.drivetrain = drivetrain

    def proportional_signal(self, KP, base_speed):
        """Generates motor speeds using proportional control bas

In [7]:
if retrieved_chunks:
    # Formulate the prompt for the LLM
    prompt = f"""You are a helpful assistant. Use the following context from class files to answer the user's question. If you don't know the answer, just say you don't know.

    Context:
    {context}

    Question: {query}"""

    # Query the LLM
    print("\nGenerating answer...")
    url = f"{OLLAMA_BASE_URL}/api/generate"
    data = {
        "model": "gemma3:4b",
        "prompt": prompt,
        "stream": False  # Set to True for streaming output
    }
    response = requests.post(url, json=data)
    response.raise_for_status()
    result = response.json()
    answer = result['response']
    if answer:
        print("\nAnswer:")
        print(answer)
    else:
        print("Could not get an answer from the LLM.")


Generating answer...

Answer:
```python
class LineSensor:
    def __init__(self):
        pass

    def left_sensor(self):
        # Replace with actual sensor readings
        return 0.2  # Simulate a left sensor reading

    def right_sensor(self):
        # Replace with actual sensor readings
        return 0.1  # Simulate a right sensor reading

class LineTracker:
    def __init__(self, drivetrain):
        self.drivetrain = drivetrain

    def proportional_signal(self, KP, base_speed):
        """Generates motor speeds using proportional control based on the error between the sensors."""
        left = self.left_sensor()
        right = self.right_sensor()
        error = left - right
        proportional_signal = KP * error
        left_motor_effort = base_speed - proportional_signal
        right_motor_effort = base_speed + proportional_signal
        return left_motor_effort, right_motor_effort

    def line_follow_proportional(self, KP, base_speed):
        """Follows the lin